In [ ]:
# optional setup
%load_ext autoreload
%autoreload 2

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'  # choose cuda-device
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"  # disable preallocation of memory

import jax
#jax.config.update("jax_platform_name", "cpu")  # optionally run on cpu

In [ ]:
from IPython.display import display, HTML
import matplotlib.pyplot as plt
import jax.numpy as jnp

from rhmag.utils.pretest_evaluation import create_multilevel_df
from rhmag.data_management import FINAL_MATERIALS, MaterialSet, DataSet, AVAILABLE_MATERIALS
from rhmag.utils.data_plotting import plot_sequence_prediction, plot_hysteresis_prediction
from rhmag.utils.model_evaluation import reconstruct_model_from_file, plot_model_frequency_sweep, evaluate_cross_validation, get_exp_ids
from rhmag.utils.final_data_evaluation import FINAL_SCENARIOS_PER_MATERIAL

---

In [ ]:
AVAILABLE_MATERIALS

In [ ]:
# get data:
data_set = DataSet.from_material_names(AVAILABLE_MATERIALS)

In [ ]:
# get models:
models = {}

for material_name in AVAILABLE_MATERIALS:
    print(material_name)
    exp_ids = get_exp_ids(material_name, exp_name="MagNetHub-reduced-features-f32")
    assert len(exp_ids) == 1
    models[material_name] = reconstruct_model_from_file(exp_ids[0])

print(models.keys())

---

# Validate models on data:
(these are known trajectories, model type is not prone to overfitting, should be okay)

In [ ]:
data_set.material_names == AVAILABLE_MATERIALS

In [ ]:
for material_set in data_set:
    print(200000 in material_set.frequencies)

In [ ]:
for material_name in data_set.material_names:
    print("MATERIAL: ", material_name)
    model = models[material_name]
    material_set = data_set.at_material(material_name)

    # choose subset of data:
    past_size = 100
    sequence_length = 500
    frequency = 500_000
    try:
        relevant_frequency_set = material_set.at_frequency(jnp.array([frequency]))
    except:
        continue
    
    B = relevant_frequency_set.B[:, :sequence_length]
    H = relevant_frequency_set.H[:, :sequence_length]
    T = relevant_frequency_set.T[:]
    
    
    H_pred = model(
        B_past=B[:, :past_size],
        B_future=B[:, past_size:],
        H_past=H[:, :past_size],
        T=T,
    )
   
    
    # visualization of predicted trajectories:
    fig, axs = plt.subplots(2, 5, figsize=(15, 5), constrained_layout=True)
    for idx in range(5):

        H_plot = H[idx]
        B_plot = B[idx]
        T_plot = T[idx]

        length = H_plot.shape[0]
        k = jnp.linspace(0, length-1, length)
        
        axs[0, idx].plot(k, B_plot, color="tab:blue")
        axs[1, idx].plot(k, H_plot, label="H_true", color="tab:blue")
        axs[1, idx].plot(k[past_size:], H_pred[idx], linestyle="--", color="tab:orange", label="H_pred")

        axs[-1, idx].set_xlabel("k")    
        axs[-1, idx].legend()
        
    for ax_ in axs:
        for ax in ax_:
            ax.grid(alpha=0.3)

    axs[0, 0].set_ylabel("B in T")
    axs[1, 0].set_ylabel("H in A/m")

    plt.show()